# When diffusion destabilises a uniform state
Two linearly coupled scalar signals evolve on a periodic ring. The operator for each spatial Fourier mode is only two by two. Compare the wavelengths selected by the continuum equation with those selected by a finite-difference grid. The model describes small deviations; it contains no nonlinear saturation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

length = 2*np.pi                  # nondimensional circumference
cells = 32                       # even number of grid cells
slow_diffusion = .02
fast_diffusion = 1.
assert 4 <= length <= 12 and cells % 2 == 0 and 16 <= cells <= 120
assert .01 <= slow_diffusion <= .3 and .2 <= fast_diffusion <= 2.
J = np.array([[1., 1.],[-3., -2.]])
D = np.diag([slow_diffusion,fast_diffusion])

def growth(q):
    q = np.asarray(q)
    trace = np.trace(J) - (slow_diffusion+fast_diffusion)*q
    det = 1 - (fast_diffusion-2*slow_diffusion)*q + slow_diffusion*fast_diffusion*q*q
    return (trace+np.sqrt((trace**2-4*det).astype(complex))).real/2

indices = np.arange(cells//2+1)
k = 2*np.pi*indices/length
h = length/cells
continuum = growth(k*k)
q_mesh = (2*np.sin(k*h/2)/h)**2
mesh = growth(q_mesh)
print("Amplifying continuum modes on this mode list:", indices[continuum>1e-10])
print("Amplifying finite-grid modes:", indices[mesh>1e-10])
assert np.allclose(mesh,[np.linalg.eigvals(J-q*D).real.max() for q in q_mesh])

## Compare a dispersion curve with an exact linear wave experiment
For each mode prepare the real part of its leading eigenvector and evolve the complex eigenmode before taking the real part. The phase and growth rates in this experiment follow the finite-grid operator exactly. The panels distinguish an incorrect spatial eigenvalue from time-stepping error. Initial amplitudes are small and intentionally comparable.

In [ ]:
fig, (curve, traceplot) = plt.subplots(1,2,figsize=(10,4),layout="constrained")
mode_range = np.linspace(0,min(14,cells//2),350)
curve.axhline(0,color="grey",lw=.8)
curve.plot(mode_range,growth((2*np.pi*mode_range/length)**2),color="#145c71",label="continuum dispersion")
visible = indices <= mode_range.max()
curve.scatter(indices[visible],mesh[visible],s=20,color="#b94626",label="mesh rates")
curve.set(xlabel="Spatial mode number",ylabel="Growth rate",title="Which wavelengths can grow?")
curve.legend(fontsize=9)
x = np.arange(cells)*h
def state_at(t):
    profile = np.zeros_like(x)
    for j in range(1,min(10,cells//2)+1):
        vals,vecs = np.linalg.eig(J-q_mesh[j]*D)
        idx = vals.real.argmax()
        vec = vecs[:,idx] / vecs[0,idx]
        profile += 1e-4*np.real(vec[0]*np.exp(vals[idx]*t+1j*(k[j]*x+.27*j*j)))
    return profile
for t,color in [(0.,"#59686a"),(3.,"#b94626"),(7.,"#145c71")]:
    traceplot.plot(x,state_at(t),color=color,label=f"t = {t:g}")
traceplot.set(xlabel="Position on ring",ylabel="First-field deviation",title="Small signals after mode selection")
traceplot.legend(fontsize=9)
for ax in (curve,traceplot): ax.grid(alpha=.2)
plt.show()

## Verify the spatial operator independently
Assemble the full periodic matrix for the selected grid. Its most rapidly growing eigenvalue should agree with the Fourier block calculation. In OneSimulation the optional comparison uses its general dense eigensolver. A small positive cut-off avoids classifying roundoff at a neutral boundary as physical growth.

In [ ]:
lap = (np.roll(np.eye(cells),1,axis=0) - 2*np.eye(cells) + np.roll(np.eye(cells),-1,axis=0))/h**2
full = np.kron(np.eye(cells),J) + np.kron(lap,D)
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import eigenvalues_dense as complete_spectrum
except ImportError:
    complete_spectrum = np.linalg.eigvals
rate = complete_spectrum(full).real.max()
assert np.isclose(rate,mesh.max(),atol=3e-10)
print(f"Full grid {rate:.8f}; fastest Fourier block {mesh.max():.8f}.")